## AVANT

In [ ]:
from trashdata import load_cifar10_from_s3

In [22]:
a,b=load_cifar10_from_s3()

/home/kouka/Projectss/MEP_2026_ALAMIDRIDEB_BOULIAIRE_KOUKA_SAMAMA/data.py:13: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(BytesIO(obj["Body"].read()), encoding="bytes")


In [ ]:
1mn

In [19]:
import os
import pickle
import numpy as np
import torch


def unpickle_local(filepath):
    with open(filepath, "rb") as f:
        return pickle.load(f, encoding="bytes")


def compute_channel_stats_uint8(X_flat_uint8):
    """
    X_flat_uint8: numpy array (N, 3072), uint8
    Calcule mean/std sur le train uniquement, après passage en [0, 1].
    """
    X = X_flat_uint8.astype(np.float64) / 255.0
    X = X.reshape(-1, 3, 32, 32)

    mean = X.mean(axis=(0, 2, 3))
    std = X.std(axis=(0, 2, 3))

    return mean.astype(np.float32), std.astype(np.float32)


def build_compact_cifar_local(
    input_dir="/mnt/c/Users/hamza/Desktop/mise en prod data/DATA/cifar-10-batches-py",
    output_path="/mnt/c/Users/hamza/Desktop/mise en prod data/DATA/processed_data/cifar10_uint8.pt"
):
    train_batches = [
        unpickle_local(os.path.join(input_dir, f"data_batch_{i}"))
        for i in range(1, 6)
    ]
    test_batch = unpickle_local(os.path.join(input_dir, "test_batch"))

    X_train = np.concatenate([batch[b"data"] for batch in train_batches], axis=0).astype(np.uint8)
    y_train = np.concatenate(
        [np.asarray(batch[b"labels"], dtype=np.int64) for batch in train_batches],
        axis=0
    )

    X_test = np.asarray(test_batch[b"data"], dtype=np.uint8)
    y_test = np.asarray(test_batch[b"labels"], dtype=np.int64)

    mean, std = compute_channel_stats_uint8(X_train)

    print("Computed train mean:", mean.tolist())
    print("Computed train std :", std.tolist())

    artifact = {
        "X_train_uint8": torch.from_numpy(X_train).contiguous(),  # (50000, 3072), uint8
        "y_train": torch.from_numpy(y_train).long().contiguous(),
        "X_test_uint8": torch.from_numpy(X_test).contiguous(),    # (10000, 3072), uint8
        "y_test": torch.from_numpy(y_test).long().contiguous(),
        "mean": torch.from_numpy(mean),
        "std": torch.from_numpy(std),
        "meta": {
            "storage_format": "uint8_flattened",
            "normalization_stats_computed_from": "train_only",
            "num_train_samples": int(X_train.shape[0]),
            "num_test_samples": int(X_test.shape[0]),
            "input_dir": input_dir,
        },
    }

    output_dir = os.path.dirname(output_path)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    torch.save(artifact, output_path)
    print(f"Compact dataset saved locally at: {output_path}")


if __name__ == "__main__":
    build_compact_cifar_local()

/tmp/ipykernel_5653/3296105.py:9: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(f, encoding="bytes")


Computed train mean: [0.4913996756076813, 0.4821584224700928, 0.44653090834617615]
Computed train std : [0.24703222513198853, 0.24348512291908264, 0.26158782839775085]
Compact dataset saved locally at: /mnt/c/Users/hamza/Desktop/mise en prod data/DATA/processed_data/cifar10_uint8.pt


In [ ]:
from io import BytesIO
import boto3
import torch
from torch.utils.data import TensorDataset, DataLoader


def preprocess_uint8_tensor(X_uint8, mean, std):
    """
    X_uint8: torch.uint8 de shape (N, 3072)
    retourne: torch.float32 de shape (N, 3, 32, 32)
    """
    X = X_uint8.float().div(255.0)
    X = X.view(-1, 3, 32, 32)

    mean = mean.view(1, 3, 1, 1).float()
    std = std.view(1, 3, 1, 1).float()

    X = (X - mean) / std
    return X


s3 = boto3.client("s3")


def load_compact_cifar_from_s3(
    bucket="data-mep-project",
    key="DATA/Cifar_10/cifar10_uint8.pt",
    batch_size=128,
    num_workers=0
):
    obj = s3.get_object(Bucket=bucket, Key=key)
    artifact = torch.load(BytesIO(obj["Body"].read()), map_location="cpu", weights_only=True)

    X_train_uint8 = artifact["X_train_uint8"]
    y_train = artifact["y_train"]
    X_test_uint8 = artifact["X_test_uint8"]
    y_test = artifact["y_test"]
    mean = artifact["mean"]
    std = artifact["std"]

    X_train = preprocess_uint8_tensor(X_train_uint8, mean, std)
    X_test = preprocess_uint8_tensor(X_test_uint8, mean, std)

    train_ds = TensorDataset(X_train, y_train)
    test_ds = TensorDataset(X_test, y_test)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available()
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available()
    )

    return train_loader, test_loader

In [25]:
load_compact_cifar_from_s3()

(<torch.utils.data.dataloader.DataLoader at 0x7ad2b12f1c50>,
 <torch.utils.data.dataloader.DataLoader at 0x7ad2b12f2450>)

In [26]:
from dataloader_cifar import load_cifar10_from_s3

train_loader, test_loader = load_cifar10_from_s3()